# SafeSign 수신호 — Evidential 모델 학습 (Colab)

공개 데이터 27,735건으로 **설정 → 구조 → 최종 모델**을 고르고, 학습한 가중치를 내려받는다.

- **KPI 데이터(자체 촬영 JH·me01)는 여기에 없다.** 최종 KPI 측정은 로컬에서 한 번만 한다.
- 작업은 **Google Drive 안**(`MyDrive/safesign_vision`)에서 한다. 세션이 끊겨도 결과가 남고,
  같은 셀을 다시 실행하면 **끝난 부분은 건너뛰고 이어서** 돈다.
- 시작 전: **런타임 → 런타임 유형 변경 → GPU** (T4 이상)

In [ ]:
# 1) GPU 확인
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch, sklearn, numpy
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| sklearn", sklearn.__version__)
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다 — 런타임 유형을 GPU 로 바꾸세요"
print(torch.cuda.get_device_name(0), "sm%d%d" % torch.cuda.get_device_capability())

## 2) Drive 연결 · 묶음 풀기

로컬에서 `python colab/make_bundle.py` 로 만든 **`safesign_colab.zip`** 을
Drive 의 `MyDrive/safesign_vision/` 에 올려 둔다. (없으면 이 셀이 업로드 창을 띄운다)

In [ ]:
import os, json, hashlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/safesign_vision"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

ZIP = os.path.join(WORK, "safesign_colab.zip")
if not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()                      # safesign_colab.zip 선택
    name = next(iter(up))
    os.replace(name, ZIP)

with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(WORK)                      # 코드·데이터만 덮어쓴다. reports/ 결과는 그대로 남는다
man = json.load(open("MANIFEST.json", encoding="utf-8"))
for f, m in man["files"].items():
    h = hashlib.sha256(open(f, "rb").read()).hexdigest()
    assert h == m["sha256"], f"파일이 손상됐습니다: {f}"
print("풀기 완료 — 파일", len(man["files"]), "개 · 공개 데이터", man["samples"], "건 · 촬영자",
      man["subjects"])

In [ ]:
# 3) 점검 — GPU 판 특징이 원본과 같은지, EDL 수식이 맞는지 (1분 이내)
!python tests/test_edl.py
!python tests/test_handformer.py

## 4) 학습 — 설정 → 구조 → 최종 (T4 기준 대략 2~2.5시간)

- **1단계 설정**: MLP 로 증거 활성 · λmax · 정규화 · 치명가중치 24조합 (각 멤버 2 + 한 클래스 빼기 7회)
- **1+ 재확인**: HandFormer-small 에서 활성 · λmax 4조합
- **2단계 구조**: MLP · MLP(증강 끔) · HandFormer small · base · **large** (각 멤버 3 + 한 클래스 빼기 7회)
- **3단계 최종**: 고른 구조로 멤버 5개 → 공개 test 1회 → `models/handformer_edl.pt`

모든 학습은 **early stopping**(val 손실, patience 20)으로 최고점에서 멈추고 그 가중치로 되돌린다.
끊기면 **이 셀을 다시 실행**하면 된다. 처음부터 다시 하려면 끝에 `--fresh` 를 붙인다.

In [ ]:
# 세션이 끊겼다 다시 붙으면 작업 폴더가 /content 로 돌아간다 — 매번 Drive 폴더로 이동한다
import os
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/safesign_vision"
os.chdir(WORK); print("작업 폴더:", os.getcwd())
!python -u training/select_model.py --profile colab

In [ ]:
# 세션이 끊겼다 다시 붙으면 작업 폴더가 /content 로 돌아간다 — 매번 Drive 폴더로 이동한다
import os
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/safesign_vision"
os.chdir(WORK); print("작업 폴더:", os.getcwd())
# 5) epoch 별 그래프
!python scripts/plot_training.py
from IPython.display import Image, display
display(Image("reports/training_curves.png"))
display(Image("reports/loco_curves.png"))

## 6) 결과 내려받기

`safesign_results.zip` 을 받아서 로컬에서

```
python colab/import_results.py 내려받은경로/safesign_results.zip
```

을 실행하면 가중치가 `services/vision/models/handformer_edl.pt` 로, 기록·그래프가
`services/vision/reports/` 로 들어가고 로컬 torch 로 읽히는지까지 확인한다.

In [ ]:
# 세션이 끊겼다 다시 붙으면 작업 폴더가 /content 로 돌아간다 — 매번 Drive 폴더로 이동한다
import os
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/safesign_vision"
os.chdir(WORK); print("작업 폴더:", os.getcwd())
import glob
out = "safesign_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("models/handformer_edl.pt")
    for f in glob.glob("reports/*.json") + glob.glob("reports/*.png"):
        zf.write(f)
print("묶음:", out, round(os.path.getsize(out) / 1e6, 1), "MB  (Drive 의 safesign_vision 폴더에도 있다)")
from google.colab import files
files.download(out)